In [10]:
library(tidyverse)
library(oce)
source('../../DATA/interpolateData.R')

# =============================================================================
# STEADY-STATE FORCING CALCULATION
# Following Stock et al. (2008) chemostat framework
#
# Depth intervals are defined here and can be adjusted per regime.
# =============================================================================

niskin_ds_3 <- readRDS("../../DATA/processed/Niskin_cleaned.rds")
CARIACO    <- readRDS("../../DATA/processed/CARIACO_EnvData_combined.rds")

# ---- Define depth intervals (adjust per regime as needed) ----
euphotic_lower <- 50
N0_upper       <- 50
N0_lower       <- 70

# ---- Extract depth-specific values using interpolateData ----

# NO3 euphotic zone mean (0-50m) — model state variable N
NO3_euphotic_ds <- interpolateData(
  niskin_ds_3, "NO3_merged",
  depth_from = 0, depth_to = euphotic_lower, noofNA = 10,
  output_type = 'mean', surface_fix = TRUE
)
names(NO3_euphotic_ds)[1] <- "NO3_euphotic"

# Export the dynamic euphotic NO3 time series for the Python plots
write.csv(NO3_euphotic_ds, "NO3_euphotic_dynamic.csv", row.names = FALSE)
cat("Saved to NO3_euphotic_dynamic.csv\n")

# NO3 sub-euphotic mean (50-70m) — model forcing N0
NO3_subeuphotic_ds <- interpolateData(
  niskin_ds_3, "NO3_merged",
  depth_from = N0_upper, depth_to = N0_lower, noofNA = 10,
  output_type = 'mean', surface_fix = FALSE
)
names(NO3_subeuphotic_ds)[1] <- "NO3_subeuphotic"

# Temperature euphotic zone mean (0-50m) — for f-ratio
Temp_euphotic_ds <- interpolateData(
  niskin_ds_3, "Temperature",
  depth_from = 0, depth_to = euphotic_lower, noofNA = 10,
  output_type = 'mean', surface_fix = TRUE
)
names(Temp_euphotic_ds)[1] <- "Temp_euphotic"

# Primary Productivity integrated (0-100m) — keep consistent with existing pipeline
PP_ds <- interpolateData(
  niskin_ds_3, "PrimaryProductivity",
  depth_from = 0, depth_to = euphotic_lower, noofNA = 20,
  output_type = 'integrated', surface_fix = TRUE
)
names(PP_ds)[1] <- "PP_integrated"

PP_ds$PP_integrated <- PP_ds$PP_integrated * 12  # hr -> day (12h tropical daylight)

# ---- Data availability ----
cat("=== Data availability ===\n")
cat(sprintf("  NO3 euphotic (0-%dm):     %d dates\n", euphotic_lower, nrow(NO3_euphotic_ds)))
cat(sprintf("  NO3 sub-euphotic (%d-%dm): %d dates\n", N0_upper, N0_lower, nrow(NO3_subeuphotic_ds)))
cat(sprintf("  Temperature (0-%dm):      %d dates\n", euphotic_lower, nrow(Temp_euphotic_ds)))
cat(sprintf("  PP integrated (0-100m):   %d dates\n", nrow(PP_ds)))

# ---- Euphotic depth from combined dataset ----
mean_euphotic_depth <- mean(CARIACO$euphotic_depth, na.rm = TRUE)

# ---- Grand means ----
mean_N0        <- mean(NO3_subeuphotic_ds$NO3_subeuphotic, na.rm = TRUE)
mean_N_surface <- mean(NO3_euphotic_ds$NO3_euphotic, na.rm = TRUE)
mean_temp      <- mean(Temp_euphotic_ds$Temp_euphotic, na.rm = TRUE)
mean_PP        <- mean(PP_ds$PP_integrated, na.rm = TRUE)

cat("\n=== Grand Mean Values ===\n")
cat(sprintf("  Euphotic depth:       %.2f m\n", mean_euphotic_depth))
cat(sprintf("  Temperature (0-%dm):  %.2f °C\n", euphotic_lower, mean_temp))
cat(sprintf("  PP (integrated):      %.2f mg C m-2 d-1\n", mean_PP))
cat(sprintf("  N0 (%d-%dm NO3):      %.4f mmol N m-3\n", N0_upper, N0_lower, mean_N0))
cat(sprintf("  N_surface (0-%dm):    %.4f mmol N m-3\n", euphotic_lower, mean_N_surface))

# ---- f-ratio (Laws et al. 2011) ----
f_ratio_calc <- function(T, PP_mgC) {
  (0.5857 - 0.0165 * T) * PP_mgC / (51.7 + PP_mgC)
}

f_ratio <- f_ratio_calc(mean_temp, mean_PP)

# ---- New production ----
new_prod_area     <- f_ratio * mean_PP                                        # mg C m-2 d-1
new_nutrient_flux <- new_prod_area / mean_euphotic_depth / 12.01 * (16 / 106) # mmol N m-3 d-1
#                                                          ^^^^^    ^^^^^^
#                                                     mg C -> mmol C  Redfield C:N

# ---- Dilution rate ----
delta_N       <- mean_N0 - mean_N_surface
dilution_rate <- new_nutrient_flux / delta_N

cat(sprintf("\n=== Derived Forcing ===\n"))
cat(sprintf("  f-ratio:              %.4f\n", f_ratio))
cat(sprintf("  New production:       %.4f mg C m-2 d-1\n", new_prod_area))
cat(sprintf("  Nutrient flux:        %.6f mmol N m-3 d-1\n", new_nutrient_flux))
cat(sprintf("  N0 - N_surface:       %.4f mmol N m-3\n", delta_N))
cat(sprintf("  Dilution rate (d):    %.6f d-1\n", dilution_rate))
cat(sprintf("  Residence time (1/d): %.1f days\n", 1 / dilution_rate))

# ---- Summary table ----
forcing_summary <- tibble(
  parameter = c("euphotic_depth", "temperature", "PP_total",
                "f_ratio", "new_production_area", "new_nutrient_flux",
                "N0", "N_surface", "N0_minus_N", "dilution_rate", "residence_time"),
  value = c(mean_euphotic_depth, mean_temp, mean_PP,
            f_ratio, new_prod_area, new_nutrient_flux,
            mean_N0, mean_N_surface, delta_N, dilution_rate, 1 / dilution_rate),
  units = c("m", "deg_C", "mg_C_m-2_d-1",
            "dimensionless", "mg_C_m-2_d-1", "mmol_N_m-3_d-1",
            "mmol_N_m-3", "mmol_N_m-3", "mmol_N_m-3", "d-1", "days"),
  depth_interval = c(
    "from_combined_dataset",
    sprintf("0-%dm", euphotic_lower),
    sprintf("0-%dm_integrated", euphotic_lower),
    "derived", "derived", "derived",
    sprintf("%d-%dm", N0_upper, N0_lower),
    sprintf("0-%dm", euphotic_lower),
    "derived", "derived", "derived")
)

print(forcing_summary)
write.csv(forcing_summary, "steady_state_forcing_summary.csv", row.names = FALSE)
cat("\nSaved to steady_state_forcing_summary.csv\n")

Saved to NO3_euphotic_dynamic.csv
=== Data availability ===
  NO3 euphotic (0-50m):     211 dates
  NO3 sub-euphotic (50-70m): 219 dates
  Temperature (0-50m):      226 dates
  PP integrated (0-100m):   219 dates

=== Grand Mean Values ===
  Euphotic depth:       44.92 m
  Temperature (0-50m):  24.34 °C
  PP (integrated):      1202.97 mg C m-2 d-1
  N0 (50-70m NO3):      5.5564 mmol N m-3
  N_surface (0-50m):    2.0158 mmol N m-3

=== Derived Forcing ===
  f-ratio:              0.1766
  New production:       212.4007 mg C m-2 d-1
  Nutrient flux:        0.059433 mmol N m-3 d-1
  N0 - N_surface:       3.5406 mmol N m-3
  Dilution rate (d):    0.016786 d-1
  Residence time (1/d): 59.6 days
# A tibble: 11 × 4
   parameter               value units          depth_interval       
   <chr>                   <dbl> <chr>          <chr>                
 1 euphotic_depth        44.9    m              from_combined_dataset
 2 temperature           24.3    deg_C          0-50m                
 3 P

In [6]:
summary(CARIACO$PrimaryProductivity)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
  2.908   7.481  10.561  13.306  16.485  77.849      52 

In [2]:
library(tidyverse)

# =============================================================================
# STEADY-STATE FORCING CALCULATION
# Following Stock et al. (2008) chemostat framework
#
# Extracts depth-specific values from raw interpolated Niskin profiles,
# then calculates N0, dilution rate, and other forcing parameters.
#
# Depth intervals can be adjusted per regime later.
# =============================================================================

profiles <- readRDS("../../DATA/processed/Niskin_forcing_profiles.rds")
CARIACO  <- readRDS("../../DATA/processed/CARIACO_EnvData_combined.rds")

# ---- Define depth intervals ----
# Adjust these per regime as needed
euphotic_upper <- 0
euphotic_lower <- 50   # approximate mean euphotic depth
N0_upper       <- 50
N0_lower       <- 70   # 20m band below euphotic zone

# ---- Helper: extract depth-layer mean per date from profiles ----
extract_layer_mean <- function(profiles, var_name, z_from, z_to) {
  profiles %>%
    filter(variable == var_name,
           depth >= z_from,
           depth <= z_to) %>%
    group_by(date) %>%
    summarise(value = mean(value_int, na.rm = TRUE), .groups = "drop") %>%
    filter(!is.nan(value))
}

# ---- Extract depth-specific values ----

# NO3 in euphotic zone (model state variable N)
NO3_euphotic <- extract_layer_mean(profiles, "NO3_merged", euphotic_upper, euphotic_lower)

# NO3 below euphotic zone (model forcing N0)
NO3_subeuphotic <- extract_layer_mean(profiles, "NO3_merged", N0_upper, N0_lower)

# Temperature in euphotic zone (for f-ratio)
Temp_euphotic <- extract_layer_mean(profiles, "Temperature", euphotic_upper, euphotic_lower)

# Primary productivity (depth-integrated, so take the full profile integral)
# PP was processed as integrated in the Niskin script, but the raw profile
# gives us depth-resolved values. We need the depth-integrated total.
PP_profiles <- profiles %>%
  filter(variable == "PrimaryProductivity",
         depth >= euphotic_upper,
         depth <= 100)  # PP integration depth from original script

# Define the total depth range for integration (100m - euphotic_upper)
integration_depth_range <- 100 - euphotic_upper

# Mean * Depth integration per date (matching legacy interpolateData.R)
PP_integrated <- PP_profiles %>%
  group_by(date) %>%
  summarise(
    PP_integrated = if (n() > 0) {
      mean(value_int, na.rm = TRUE) * integration_depth_range
    } else {
      NA_real_
    },
    .groups = "drop"
  ) %>%
  filter(!is.na(PP_integrated))

# Apply unit conversion (same as in data script: * 12 for g C -> mg C)
PP_integrated$PP_integrated <- PP_integrated$PP_integrated * 12

cat("=== Data availability ===\n")
cat(sprintf("  NO3 euphotic (0-%dm):     %d dates\n", euphotic_lower, nrow(NO3_euphotic)))
cat(sprintf("  NO3 sub-euphotic (%d-%dm): %d dates\n", N0_upper, N0_lower, nrow(NO3_subeuphotic)))
cat(sprintf("  Temperature (0-%dm):      %d dates\n", euphotic_lower, nrow(Temp_euphotic)))
cat(sprintf("  PP integrated:            %d dates\n", nrow(PP_integrated)))

# ---- Euphotic depth from combined dataset ----
mean_euphotic_depth <- mean(CARIACO$euphotic_depth, na.rm = TRUE)

# ---- Grand means ----
mean_N0        <- mean(NO3_subeuphotic$value, na.rm = TRUE)
mean_N_surface <- mean(NO3_euphotic$value, na.rm = TRUE)
mean_temp      <- mean(Temp_euphotic$value, na.rm = TRUE)
mean_PP        <- mean(PP_integrated$PP_integrated, na.rm = TRUE)

cat("\n=== Grand Mean Values ===\n")
cat(sprintf("  Euphotic depth:       %.2f m\n", mean_euphotic_depth))
cat(sprintf("  Temperature (0-%dm):  %.2f °C\n", euphotic_lower, mean_temp))
cat(sprintf("  PP (integrated):      %.2f mg C m-2 d-1\n", mean_PP))
cat(sprintf("  N0 (%d-%dm NO3):      %.4f mmol N m-3\n", N0_upper, N0_lower, mean_N0))
cat(sprintf("  N_surface (0-%dm):    %.4f mmol N m-3\n", euphotic_lower, mean_N_surface))

# ---- f-ratio (Laws et al. 2011) ----
f_ratio_calc <- function(T, PP_mgC) {
  (0.5857 - 0.0165 * T) * PP_mgC / (51.7 + PP_mgC)
}

f_ratio <- f_ratio_calc(mean_temp, mean_PP)

# ---- New production ----
new_prod_area    <- f_ratio * mean_PP                                      # mg C m-2 d-1
new_nutrient_flux <- new_prod_area / mean_euphotic_depth / 12.01 * (16/106) # mmol N m-3 d-1

# ---- Dilution rate ----
delta_N       <- mean_N0 - mean_N_surface
dilution_rate <- new_nutrient_flux / delta_N

cat(sprintf("\n=== Derived Forcing ===\n"))
cat(sprintf("  f-ratio:              %.4f\n", f_ratio))
cat(sprintf("  New production:       %.4f mg C m-2 d-1\n", new_prod_area))
cat(sprintf("  Nutrient flux:        %.6f mmol N m-3 d-1\n", new_nutrient_flux))
cat(sprintf("  N0 - N_surface:       %.4f mmol N m-3\n", delta_N))
cat(sprintf("  Dilution rate (d):    %.6f d-1\n", dilution_rate))
cat(sprintf("  Residence time (1/d): %.1f days\n", 1 / dilution_rate))

# ---- Summary table ----
forcing_summary <- tibble(
  parameter = c("euphotic_depth", "temperature", "PP_total",
                "f_ratio", "new_production_area", "new_nutrient_flux",
                "N0", "N_surface", "N0_minus_N", "dilution_rate", "residence_time"),
  value = c(mean_euphotic_depth, mean_temp, mean_PP,
            f_ratio, new_prod_area, new_nutrient_flux,
            mean_N0, mean_N_surface, delta_N, dilution_rate, 1/dilution_rate),
  units = c("m", "deg_C", "mg_C_m-2_d-1",
            "dimensionless", "mg_C_m-2_d-1", "mmol_N_m-3_d-1",
            "mmol_N_m-3", "mmol_N_m-3", "mmol_N_m-3", "d-1", "days"),
  depth_interval = c(
    "from_combined_dataset", 
    sprintf("0-%dm", euphotic_lower), 
    "0-100m_integrated",
    "derived", "derived", "derived",
    sprintf("%d-%dm", N0_upper, N0_lower),
    sprintf("0-%dm", euphotic_lower),
    "derived", "derived", "derived")
)

print(forcing_summary)
#write.csv(forcing_summary, "processed/steady_state_forcing_summary.csv", row.names = FALSE)
#cat("\nSaved to processed/steady_state_forcing_summary.csv\n")

ERROR: [1m[33mError[39m in `summarise()`:[22m
[1m[22m[36mℹ[39m In argument: `PP_integrated = if (...) NULL`.
[36mℹ[39m In group 1: `date = 1995-12-13`.
[1mCaused by error in `loadNamespace()`:[22m
[33m![39m es gibt kein Paket namens ‘pracma’
